<a href="https://colab.research.google.com/github/Pramodsingh317/E-Commerce-Customer-Churn-Prediction-and-Retention-System/blob/main/E_Commerce_Customer_Churn_Prediction_and_Retention_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# MODULE 1: SYSTEM INITIALIZATION & PACKAGES
# ==============================================================================
!pip install xgboost shap lifetimes streamlit plotly pyngrok imbalanced-learn -q

import json
import subprocess
import time
import os
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import plotly.express as px
import shap
import streamlit as st
import xgboost as xgb
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

print("Module 1 Complete: Dependencies successfully verified and imported.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.2/584.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 63.2 MB/s eta 0:00:00
Module 1 Complete: Dependencies successfully verified and imported.


In [2]:
# ==============================================================================
# MODULE 2: PROGRAMMATIC SYNTHETIC DATA GENERATION ENGINE (RESTRUCTURED)
# ==============================================================================
np.random.seed(42)
TOTAL_CUSTOMERS = 500
customer_pool = [f"CUST_{str(i).zfill(3)}" for i in range(1, TOTAL_CUSTOMERS + 1)]
START_DATE = datetime(2026, 1, 1)

# Available product categories for recommendations
PRODUCT_CATEGORIES = ['Electronics', 'Apparel', 'Home & Kitchen', 'Beauty', 'Sports & Outdoors']

# --- 1. Transaction Generation ---
transaction_logs = []
for customer_id in customer_pool:
    purchase_count = np.random.randint(1, 12)
    for _ in range(purchase_count):
        days_offset = np.random.randint(0, 150)
        transaction_logs.append({
            "transaction_id": f"TXN_{np.random.randint(10000, 99999)}",
            "customer_id": customer_id,
            "purchase_date": START_DATE + timedelta(days=days_offset),
            "order_amount": round(np.random.uniform(20.0, 250.0), 2)
        })
df_transactions = pd.DataFrame(transaction_logs)

# --- 2. Web Session Log Generation (RESTRUCTURED) ---
web_session_logs = []
for customer_id in customer_pool:
    session_count = np.random.randint(2, 15)
    for _ in range(session_count):
        days_offset = np.random.randint(0, 150)
        web_session_logs.append({
            "session_id": f"SESS_{np.random.randint(10000, 99999)}",
            "customer_id": customer_id,
            "session_timestamp": START_DATE + timedelta(days=days_offset),
            "session_duration_secs": np.random.randint(30, 1200),
            "product_views": np.random.randint(1, 15),
            "cart_abandoned": np.random.choice([0, 1], p=[0.8, 0.2]),
            "last_viewed_category": np.random.choice(PRODUCT_CATEGORIES) # Added for recommendations
        })
df_web_logs = pd.DataFrame(web_session_logs)

# --- 3. Customer Support Ticket Generation ---
ticket_logs = []
for customer_id in customer_pool:
    if np.random.rand() > 0.4:
        ticket_count = np.random.randint(1, 4)
        for _ in range(ticket_count):
            days_offset = np.random.randint(0, 150)
            ticket_logs.append({
                "ticket_id": f"TKT_{np.random.randint(10000, 99999)}",
                "customer_id": customer_id,
                "created_date": START_DATE + timedelta(days=days_offset),
                "resolution_time_hours": round(np.random.uniform(0.5, 48.0), 1),
                "customer_sentiment": np.random.choice(['Positive', 'Neutral', 'Negative'], p=[0.2, 0.3, 0.5])
            })
df_tickets = pd.DataFrame(ticket_logs)

print("Module 2 Complete: Synthetic logs populated with Product Category tracking.")
print(f"Transactions: {df_transactions.shape[0]} | Web Logs: {df_web_logs.shape[0]} | Tickets: {df_tickets.shape[0]}")


Module 2 Complete: Synthetic logs populated with Product Category tracking.
Transactions: 3046 | Web Logs: 3910 | Tickets: 603


In [3]:
# ==============================================================================
# MODULE 3: DATA ENGINEERING & ETL PIPELINE
# ==============================================================================
last_operational_date = df_transactions['purchase_date'].max()

# Aggregate Behavioral Core Profiles (RFM)
rfm_profiles = df_transactions.groupby('customer_id').agg(
    last_purchase=('purchase_date', 'max'),
    frequency=('transaction_id', 'count'),
    monetary=('order_amount', 'sum')
).reset_index()

rfm_profiles['frequency'] = rfm_profiles['frequency'] - 1
rfm_profiles['recency'] = (last_operational_date - rfm_profiles['last_purchase']).dt.days
rfm_profiles['churn_label'] = np.where(rfm_profiles['recency'] > 30, 1, 0)

# Aggregate Web Session Features (Extracting dominant category interest)
fav_category = df_web_logs.groupby('customer_id')['last_viewed_category'].agg(lambda x: x.mode()[0]).reset_index()
fav_category.columns = ['customer_id', 'preferred_category']

web_features = df_web_logs.groupby('customer_id').agg(
    avg_session_duration=('session_duration_secs', 'mean'),
    total_product_views=('product_views', 'sum'),
    total_cart_abandoned=('cart_abandoned', 'sum')
).reset_index()
web_features = web_features.merge(fav_category, on='customer_id', how='left')

# Map and Aggregate Support Ticket Sentiment Features
sentiment_weights = {'Positive': 1, 'Neutral': 0, 'Negative': -1}
df_tickets['sentiment_score'] = df_tickets['customer_sentiment'].map(sentiment_weights)
ticket_features = df_tickets.groupby('customer_id').agg(
    total_tickets=('ticket_id', 'count'),
    avg_resolution_time=('resolution_time_hours', 'mean'),
    avg_sentiment=('sentiment_score', 'mean')
).reset_index()

# Consolidate Unified Analytical Feature Matrix
features_master_df = rfm_profiles.merge(web_features, on='customer_id', how='left')
features_master_df = features_master_df.merge(ticket_features, on='customer_id', how='left')

fill_zero_cols = [
    'avg_session_duration', 'total_product_views', 'total_cart_abandoned',
    'total_tickets', 'avg_resolution_time', 'avg_sentiment'
]
features_master_df[fill_zero_cols] = features_master_df[fill_zero_cols].fillna(0)
features_master_df['preferred_category'] = features_master_df['preferred_category'].fillna('General')

ml_ready_features = features_master_df.drop(columns=['last_purchase'])
print(f"Module 3 Complete: Production Matrix Verified. Shape: {ml_ready_features.shape}")


Module 3 Complete: Production Matrix Verified. Shape: (500, 12)


In [4]:
# ==============================================================================
# MODULE 4: ML MODEL TRAINING, EXPLAINABLE AI, & STRATIFICATION
# ==============================================================================
# Drop non-numeric and target columns for training
X = ml_ready_features.drop(columns=['customer_id', 'churn_label', 'preferred_category'])
y = ml_ready_features['churn_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

smote_balancer = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote_balancer.fit_resample(X_train, y_train)

# Fit Churn Classification Model
churn_classifier = xgb.XGBClassifier(random_state=42, eval_metric='logloss')
churn_classifier.fit(X_train_balanced, y_train_balanced)

# Calculate Core Explainable Drivers via SHAP Tree Explainer
shap_tree_explainer = shap.TreeExplainer(churn_classifier)
calculated_shap_values = shap_tree_explainer.shap_values(X)
feature_headers = X.columns.tolist()

primary_risk_drivers = [
    feature_headers[np.argmax(customer_shap_vector)]
    for customer_shap_vector in calculated_shap_values
]

# Calculate and Predict Customer Lifetime Value (CLV)
clv_target_proxy = (ml_ready_features['monetary'] * ml_ready_features['frequency']) / (ml_ready_features['recency'] + 1)
clv_regressor = RandomForestRegressor(n_estimators=50, random_state=42)
clv_regressor.fit(X, clv_target_proxy)
predicted_clv = clv_regressor.predict(X)

# Construct Consolidated Inference Dataset
compiled_inference_df = pd.DataFrame({
    'customer_id': ml_ready_features['customer_id'],
    'preferred_category': ml_ready_features['preferred_category'],
    'churn_probability': churn_classifier.predict_proba(X)[:, 1],
    'primary_churn_reason': primary_risk_drivers,
    'clv': predicted_clv
})

# Risk Stratification Matrix
compiled_inference_df['clv_tier'] = pd.qcut(compiled_inference_df['clv'], q=2, labels=['Low Value', 'High Value'])
compiled_inference_df['risk_tier'] = np.where(compiled_inference_df['churn_probability'] >= 0.70, 'High Risk', 'Low Risk')
compiled_inference_df['segment'] = compiled_inference_df['clv_tier'].astype(str) + ", " + compiled_inference_df['risk_tier']

print("Module 4 Complete: ML predictions and features compiled successfully.")


Module 4 Complete: ML predictions and features compiled successfully.


In [5]:
# ==============================================================================
# MODULE 5: AUTOMATED RETENTION CAMPAIGN ENGINE (NEW EXECUTION LAYER)
# ==============================================================================
def execute_automated_retention_campaign(inference_df):
    """
    Automated Campaign Trigger Engine: Iterates through predictive matrices,
    maps specific root drivers to targeted incentives, and executes recommendations.
    """
    # 1. AUTOMATED TRIGGERING: Isolate users exceeding risk thresholds
    high_risk_segment = inference_df[inference_df['churn_probability'] >= 0.70].copy()

    # Campaign Mapping Inventories
    OFFER_MATRIX = {
        'recency': 'We miss you! Get a 25% discount on your next checkout with code: RETURN25',
        'total_cart_abandoned': 'Items are waiting! Free Shipping applied to your cart. Use code: FREESHIP',
        'avg_sentiment': 'We want to make things right. A dedicated account manager is assigned to you. Reply for 1-on-1 support.',
        'total_tickets': 'Priority Support Unlock! Your next support request goes directly to our executive team.',
        'frequency': 'Loyalty Reward! Claim a free gift worth $30 on your next order using code: VIPGIFT',
        'avg_session_duration': 'Exclusive Access! Check out our new curated collection tailored just for you.'
    }

    RECOMMENDATION_MATRIX = {
        'Electronics': 'Top picks: Wireless ANC Earbuds, Ultra-Slim Power Bank',
        'Apparel': 'Top picks: Premium Cotton Hoodie, Breathable Running Shoes',
        'Home & Kitchen': 'Top picks: Ergonomic Espresso Maker, Smart LED Ambient Bar',
        'Beauty': 'Top picks: Hydrating Serum Blend, Organic Clay Detox Mask',
        'Sports & Outdoors': 'Top picks: Lightweight Hydration Pack, All-Weather Trail Compass',
        'General': 'Top picks: Best Sellers Showcase Collection'
    }

    campaign_logs = []

    print(f"⚙️ Automated Trigger Init: Processing {len(high_risk_segment)} accounts meeting risk criteria...")

    # 2 & 3. PERSONALIZED OFFERS & PRODUCT RECOMMENDATIONS MAPPER
    for _, customer in high_risk_segment.iterrows():
        cust_id = customer['customer_id']
        reason = customer['primary_churn_reason']
        category = customer['preferred_category']

        # Fallback handling for missing keys
        offer = OFFER_MATRIX.get(reason, "Enjoy a $15 store credit on us with code: THANKYOU")
        recommendation = RECOMMENDATION_MATRIX.get(category, RECOMMENDATION_MATRIX['General'])

        # 4. RE-ENGAGEMENT EMAIL SIMULATION API
        email_payload = {
            "recipient_email": f"{cust_id.lower()}@enterprise-client-pool.com",
            "subject": "A Special Update Regarding Your Account Status",
            "body_incentive": offer,
            "body_recommendations": recommendation,
            "timestamp_dispatched": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "status": "SENT_SUCCESS"
        }
        campaign_logs.append(email_payload)

    df_campaign_dispatch = pd.DataFrame(campaign_logs)
    return df_campaign_dispatch

# Execute engine and cache data for the final presentation dashboard UI
df_dispatched_emails = execute_automated_retention_campaign(compiled_inference_df)

# Merge back results to preserve a unified dashboard file state
retention_summary_df = compiled_inference_df.merge(
    df_dispatched_emails[['recipient_email', 'body_incentive', 'body_recommendations']],
    left_on=compiled_inference_df['customer_id'].apply(lambda x: f"{x.lower()}@enterprise-client-pool.com"),
    right_on='recipient_email',
    how='left'
).drop(columns=['recipient_email'])

retention_summary_df.to_csv('dashboard_data.csv', index=False)
print(f"Module 5 Complete: Retention loop processed. Saved logs for {df_dispatched_emails.shape[0]} automated dispatches.")


⚙️ Automated Trigger Init: Processing 158 accounts meeting risk criteria...
Module 5 Complete: Retention loop processed. Saved logs for 158 automated dispatches.


In [6]:
# ==============================================================================
# MODULE 6: WRITE PRESENTATION DASHBOARD WEB CODE TO SYSTEM (SECURED)
# ==============================================================================
with open("app.py", "w") as web_app_file:
    web_app_file.write("""
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(layout="wide", page_title="ML Churn Platform", page_icon="📊")
st.title("📊 Enterprise ML Customer Churn Analytics & Retention Platform")
st.markdown("##### Real-Time Predictive Operational Control Dashboard")

df = pd.read_csv('dashboard_data.csv')

# --- LOCK SCREEN: BASIC AUTHENTICATION LAYER ---
if "authenticated" not in st.session_state:
    st.session_state["authenticated"] = False

if not st.session_state["authenticated"]:
    password = st.text_input("Enter Enterprise Access Key:", type="password")
    if password == "april2026":
        st.session_state["authenticated"] = True
        st.rerun()
    else:
        if password: st.error("Invalid Credentials.")
        st.stop()

# --- SECTION 1: TOP-LINE SYSTEM METRICS & REVENUE IMPACT ---
high_risk_segment = df[df['churn_probability'] >= 0.70]
total_flagged_users = len(high_risk_segment)
simulated_conversions = int(total_flagged_users * 0.18)
avg_clv_high_risk = high_risk_segment['clv'].mean() if total_flagged_users > 0 else 0
estimated_financial_savings = simulated_conversions * avg_clv_high_risk

kpi_col1, kpi_col2, kpi_col3 = st.columns(3)
with kpi_col1:
    st.metric(label="🚩 High-Risk Churners Identified", value=f"{total_flagged_users} Users", delta="Risk Prob >= 70%")
with kpi_col2:
    st.metric(label="🔄 Simulated Saved Customers", value=f"{simulated_conversions} Accounts", delta="18% Target Rate")
with kpi_col3:
    st.metric(label="💰 Projected Retained Revenue (ML CLV)", value=f"${estimated_financial_savings:,.2f}", delta="Savings Proxy")

st.markdown("---")

# --- SECTION 2: GRAPHICAL DISTRIBUTION PLOTS ---
left_chart_col, right_chart_col = st.columns(2)
with left_chart_col:
    st.subheader("👥 Customer Strategic Segmentation Cross-Section")
    segment_distribution = df['segment'].value_counts().reset_index()
    segment_distribution.columns = ['Strategic Quadrant Group', 'Customer Count']
    fig_segment_bar = px.bar(
        segment_distribution, x='Strategic Quadrant Group', y='Customer Count',
        color='Strategic Quadrant Group', template="none", text_auto=True
    )
    st.plotly_chart(fig_segment_bar, use_container_width=True)

with right_chart_col:
    st.subheader("💡 Churn Driver Core Causation Diagnostics (SHAP AI)")
    root_cause_distribution = df['primary_churn_reason'].value_counts().reset_index()
    root_cause_distribution.columns = ['Root Cause Factor', 'Account Frequency Count']
    fig_reason_pie = px.pie(
        root_cause_distribution, names='Root Cause Factor', values='Account Frequency Count',
        hole=0.4, template="none"
    )
    st.plotly_chart(fig_reason_pie, use_container_width=True)

st.markdown("---")

# --- SECTION 3: GRANULAR ENTERPRISE RISK & AUTOMATED RETENTION LOG LOOKUP ---
st.subheader("🔍 Individual Customer Risk Profile & Automated Campaign Index")
search_query = st.text_input("Enter exact Customer ID to pull profile file logs (e.g., CUST_045):").strip().upper()

filtered_profiles_df = df[df['customer_id'].astype(str).str.contains(search_query)] if search_query else df

# Interactive Dataframe Index Selection
display_columns_df = filtered_profiles_df[['customer_id', 'churn_probability', 'primary_churn_reason', 'preferred_category', 'clv']]
display_columns_df.columns = ['Customer ID Identifier', 'Model Churn Risk Score', 'SHAP Churn Root Driver', 'Preferred Browsing Category', 'Predicted Annual CLV ($)']

st.dataframe(
    display_columns_df.style.format({
        'Model Churn Risk Score': '{:.2%}',
        'Predicted Annual CLV ($)': '${:,.2f}'
    }),
    use_container_width=True
)

# Show detailed re-engagement cards if look-up returns active context
if search_query and not filtered_profiles_df.empty:
    target_row = filtered_profiles_df.iloc[0]
    st.markdown("### ✉️ Automated Re-Engagement Campaign Dispatched Summary")
    if target_row['churn_probability'] >= 0.70:
        c1, c2 = st.columns(2)
        with c1:
            st.info(f"**🎯 Triggered Dynamic Promotion:**\\n\\n {target_row['body_incentive']}")
        with c2:
            st.success(f"**📦 Personalized Product Recommendation:**\\n\\n {target_row['body_recommendations']}")
    else:
        st.write("💡 *This customer is currently marked Low Risk. Automated re-engagement scripts remain paused until thresholds are crossed.*")
""")
print("Module 6 Complete: Secured 'app.py' script layout has been updated on local disk.")


Module 6 Complete: Secured 'app.py' script layout has been updated on local disk.


In [ ]:
import subprocess
import time

print("Stopping overlapping active instances...")
!pkill -f streamlit
!pkill -f ssh

print("Instantiating Streamlit engine architecture...")
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.address", "127.0.0.1",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Extended waiting time to prevent 502 Bad Gateway racing conditions
print("Waiting for Streamlit server initialization...")
time.sleep(8)

print("\n🔗 INTERACTIVE APPLICATION PIPELINE REDEPLOYED:")
print("👉 Click the tunnel link down below to access your dynamic platform layout.")
print("🔑 Password == 'april2026'\n")

!ssh -o StrictHostKeyChecking=no -T -p 443 -R 80:127.0.0.1:8501 a.pinggy.io


Stopping overlapping active instances...
Instantiating Streamlit engine architecture...
Waiting for Streamlit server initialization...

🔗 INTERACTIVE APPLICATION PIPELINE REDEPLOYED:
👉 Click the tunnel link down below to access your dynamic platform layout.
🔑 Password == 'april2026'

You are not authenticated.
Your tunnel will expire in 60 minutes. Upgrade to Pinggy Pro to get unrestricted tunnels. https://dashboard.pinggy.io
http://fikfe-35-227-120-251.run.pinggy-free.link
https://fikfe-35-227-120-251.run.pinggy-free.link
RB: 43011, SB: 10021660, TC: 18, AC: 0               

In [11]:
!pkill -f streamlit
!pkill -f ssh
print("Network ports cleared successfully.")


Network ports cleared successfully.
